# 15 · GRU re-tune on the initiation split, with long-range context and an input shortcut (E15)

E14 picked the GRU on `fixed_plus` (val MSE 0.1784, IC 0.199). Its settings came from E4, tuned on the old split,
whose validation overlaps the new test. This notebook re-tunes it on the **new validation period** and lets the
search try two architecture changes; validation decides whether they stay:

* **Long-range context** (`inputs = fixed_plus_long`): +18 slow, causal inputs (4-hour and 1-day returns,
  trend gaps, volatility and moving averages of order flow, volume, close-vs-VWAP, ...; `rltrade.long_context`),
  plus windows up to 2048 minutes. The model predicts up to 900 minutes ahead but saw at most 256.
* **Input shortcut** (`skip = True`): a linear path from the current bar's inputs to the 8 outputs, next to the
  GRU (`rnn.zoo`), starting at zero.

Then the best configuration is retrained over 3 seeds, and **averaged predictions** are scored: the mean of the
seeds, and a blend with E14's trees. Lowest validation MSE wins, as in E4 / E14. **Test and stress_bear are
predicted and saved, never shown here.**

Input: `thesis_rl/initiation/initiation_prepared.zip` (already on Drive from E14) and E14's runs in
`thesis_rl/initiation/factors/`. Runs resume: the Optuna study and finished runs are kept on Drive.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/14-initiation"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "15_gru_retune")

In [ ]:
import json, zipfile, numpy as np, pandas as pd
from pathlib import Path
from rltrade import long_context
from rltrade.rnn import tune_gru as TG
from rltrade.rnn.train import label_metrics, available_splits

INIT = L.root / "initiation"
E14, OUT, REP = INIT / "factors", INIT / "e15", INIT / "report"
for p in (OUT, REP): p.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/initiation_prepared") if "google.colab" in sys.modules else INIT / "prepared"
ZIP = INIT / "initiation_prepared.zip"
if not (LOCAL / "fixed_plus" / "meta.json").exists():
    assert ZIP.exists(), f"upload initiation_prepared.zip to {ZIP}"
    print("unzipping", ZIP, "->", LOCAL)
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL)
if not (LOCAL / "fixed_plus_long" / "meta.json").exists():
    long_context.build(LOCAL / "fixed_plus", LOCAL / "fixed_plus_long")
PREP = {s: LOCAL / s for s in TG.INPUT_SETS}
SPLITS = available_splits(PREP["fixed_plus"])
Y_VAL = np.load(PREP["fixed_plus"] / "labels_val.npy")
E14_GRU = [E14 / f"gru_fixed_plus_s{s}" for s in (0, 1, 2)]
E14_GBDT = [E14 / f"gbdt_fixed_plus_s{s}" for s in (0, 1, 2)]
N_TRIALS, TRIAL_STEPS, TRIAL_MINUTES = 30, 3000, 8
SEEDS = (0, 1, 2)
def short(m): return {"mse": round(m["mse"], 5), "ic": round(m["ic_mean"], 4), "dir_acc": round(m["dir_acc_mean"], 4)}

## 1 · Averaging E14's models (no training)

In [ ]:
rows = [{"what": f"E14 GRU fixed_plus, seed {s}", **short(label_metrics(np.load(d / 'factors_val.npy'), Y_VAL))}
        for s, d in enumerate(E14_GRU)]
rows.append({"what": "E14 GRU fixed_plus, mean of 3 seeds", **short(TG.ensemble(E14_GRU, Y_VAL))})
rows.append({"what": "E14 trees fixed_plus, mean of 3 seeds", **short(TG.ensemble(E14_GBDT, Y_VAL))})
w14, grid = TG.blend(E14_GRU, E14_GBDT, Y_VAL)
b = min(grid, key=lambda r: r["mse"])
rows.append({"what": f"E14 GRU x {w14} + trees x {1 - w14:.1f}", "mse": round(b["mse"], 5), "ic": round(b["ic"], 4), "dir_acc": round(b["dir_acc"], 4)})
avg14 = pd.DataFrame(rows); print(avg14.to_string(index=False))
print("\nblend grid (weight on the GRU):\n" + pd.DataFrame(grid).round(5).to_string(index=False))

## 2 · Re-tune (Optuna, about 1-4 min per trial; the study resumes after a disconnect)

In [ ]:
study = TG.tune(PREP, OUT / "tuning", n_trials=N_TRIALS, max_steps=TRIAL_STEPS, max_minutes=TRIAL_MINUTES)
tr = study.trials_dataframe()
ok = tr[tr.state == "COMPLETE"].sort_values("value")
cols = ["number", "value", "user_attrs_val_ic_mean", "params_inputs", "params_skip", "params_seq_len", "params_mid_dim",
        "params_num_layers", "params_dropout", "params_lr", "params_optimizer", "params_batch_size", "user_attrs_minutes"]
print(f"{len(ok)} complete, {(tr.state == 'PRUNED').sum()} pruned")
print(ok[cols].head(10).round(5).to_string(index=False))
print("\nbest validation MSE by option (complete trials):")
for c in ("params_inputs", "params_skip", "params_seq_len"):
    print(ok.groupby(c).value.agg(["min", "median", "count"]).round(5).to_string(), "\n")
BEST = study.best_trial.params
print("best:", BEST)
(OUT / "best_params.json").write_text(json.dumps(BEST, indent=1, default=str))

## 3 · Best configuration, full budget, 3 seeds

In [ ]:
BEST = json.loads((OUT / "best_params.json").read_text())
inputs, FINAL = TG.final_runs(BEST, PREP, OUT, seeds=SEEDS)
print("inputs:", inputs, "|", [d.name for d in FINAL])

## 4 · Comparison on validation and selection

In [ ]:
cands = {}
for s, d in zip(SEEDS, FINAL):
    cands[f"tuned GRU, seed {s}"] = label_metrics(np.load(d / "factors_val.npy"), Y_VAL)
single_mean = {k: np.mean([cands[f"tuned GRU, seed {s}"][k] for s in SEEDS]) for k in ("mse", "ic_mean", "dir_acc_mean")}
table = [{"what": "E14 GRU fixed_plus (mean over seeds)", "mse": avg14.mse[:3].mean(), "ic": avg14.ic[:3].mean(), "dir_acc": avg14.dir_acc[:3].mean()},
         {"what": "tuned GRU (mean over seeds)", "mse": single_mean["mse"], "ic": single_mean["ic_mean"], "dir_acc": single_mean["dir_acc_mean"]},
         {"what": "tuned GRU, average of 3 seeds", **short(TG.ensemble(FINAL, Y_VAL))}]
w15, grid15 = TG.blend(FINAL, E14_GBDT, Y_VAL)
b = min(grid15, key=lambda r: r["mse"])
table.append({"what": f"tuned GRU average x {w15} + E14 trees x {1 - w15:.1f}", "mse": b["mse"], "ic": b["ic"], "dir_acc": b["dir_acc"]})
cmp = pd.DataFrame(table).round(5)
print(cmp.to_string(index=False))
print("\nNote: the blend weight is chosen on validation and scored on it, so its row is slightly optimistic.")

# selection: lowest validation MSE among the candidates the agent could use
opts = {"tuned GRU, best single seed": None, "tuned GRU, average of 3 seeds": cmp.mse[2], "tuned GRU + trees blend": cmp.mse[3]}
best_seed = min(SEEDS, key=lambda s: cands[f"tuned GRU, seed {s}"]["mse"])
opts["tuned GRU, best single seed"] = cands[f"tuned GRU, seed {best_seed}"]["mse"]
choice = min(opts, key=opts.get)
if choice == "tuned GRU, best single seed":
    fdir = FINAL[SEEDS.index(best_seed)]
elif choice == "tuned GRU, average of 3 seeds":
    fdir = TG.write_mixture(FINAL, [], 1.0, OUT / "selected_mean3", SPLITS)
else:
    fdir = TG.write_mixture(FINAL, E14_GBDT, w15, OUT / "selected_blend", SPLITS)
sel = {"choice": choice, "val_mse": float(opts[choice]), "factor_dir": str(fdir), "inputs": inputs, "params": BEST,
       "rule": "lowest validation MSE among: best single seed, mean of 3 seeds, blend with E14 trees (weight on val)"}
(INIT / "selected_factors_e15.json").write_text(json.dumps(sel, indent=1, default=str))
print("\nselected:", choice, "->", fdir)
(REP / "e15_gru_retune.md").write_text("# E15 GRU re-tune, validation\n\n## Averaging E14's models\n\n" + avg14.to_markdown(index=False)
    + "\n\n## Best trials\n\n" + ok[cols].head(10).round(5).to_markdown(index=False)
    + "\n\n## Comparison\n\n" + cmp.to_markdown(index=False) + f"\n\nSelected: {choice} ({fdir})\n", encoding="utf-8")
print("wrote", REP / "e15_gru_retune.md", "and", INIT / "selected_factors_e15.json")